# 🏥 Medical Report Generation - Kaggle'da Kaldığı Yerden Eğitime Devam Etme (RESUME)
Yerel bilgisayarda alınan OOM (Memory Error) hatasını Kaggle üzerinde düşük Batch Size + Gradient Accumulation ile aşarak eğitime kaldığımız yerden (Epoch 3 sonrası, Unfreeze aşaması) devam ediyoruz.

## 1️⃣ Yerel Model Checkpoint'ini Kaggle'a Yükleme
Kendi PC'nde bulunan `checkpoints_transformer/best_model` (veya `checkpoint_epoch_X`) klasöründeki dosyaları (bin, pth vs.) Kaggle'da yeni bir Dataset olarak yüklemelisin. (Sağ paneldeki **Add Data -> New Dataset** butonunu kullan, örneğin dataset adına `my-transformer-checkpoint` de).
Yükledikten sonra o datasetin Kaggle üzerindeki dosya yolunu (örneğin `/kaggle/input/my-transformer-checkpoint/best_model`) aşağıdaki hücrede `LOCAL_CHECKPOINT_DIR` olarak gir.

In [ ]:
!pip install -q transformers>=4.30.0 rouge-score pycocoevalcap

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import os
if not os.path.exists('/kaggle/working/medical-report-generation-cxr'):
    !git clone https://github.com/MustafaYagizKaragoz/medical-report-generation-cxr.git /kaggle/working/repo
else:
    import shutil
    shutil.rmtree('/kaggle/working/repo', ignore_errors=True)
    !git clone https://github.com/MustafaYagizKaragoz/medical-report-generation-cxr.git /kaggle/working/repo

os.chdir('/kaggle/working/repo')
print('Çalışma dizini:', os.getcwd())

In [ ]:
# Veri yollarını ayarla (Kaggle dataset yollarına göre güncelle)
# Kaggle'a yüklediğin dataset'in adına göre değiştir

TRAIN_CSV  = '/kaggle/input/mimic-cxr-processed/train_processed.csv'
VAL_CSV    = '/kaggle/input/mimic-cxr-processed/val_processed.csv'
TEST_CSV   = '/kaggle/input/mimic-cxr-processed/test_processed.csv'
IMAGE_DIR  = '/kaggle/input/mimic-cxr-images/files'  # Görüntü klasörü

print("📥 Local Dataset kopyalanıyor...")
import shutil
if not os.path.exists('/kaggle/working/mimic-cxr-images'):
    try: 
        shutil.copytree(IMAGE_DIR, '/kaggle/working/mimic-cxr-images')
        IMAGE_DIR = '/kaggle/working/mimic-cxr-images'
    except:
        print("Kopyalama atlandı, doğrudan okunacak.")
else:
    IMAGE_DIR = '/kaggle/working/mimic-cxr-images'

# ⚠️ AŞAĞIDAKİ YOLU KENDİ YÜKLEDİĞİN CHECKPOINT KLASÖRÜNE GÖRE DÜZENLE!
# (İçinde pytorch_model.bin, config.json, training_state.pth vb. dosyalar olan klasör)
LOCAL_CHECKPOINT_DIR = '/kaggle/input/my-transformer-checkpoint' 

print("Kontrol ediliyor...")
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV, LOCAL_CHECKPOINT_DIR]:
    exists = os.path.exists(path)
    print(f"{str('✅') if exists else str('❌')} {path}")

In [ ]:
# Config'i Kaggle ortamına ve RESUME moduna göre güncelle
import sys
sys.path.insert(0, '/kaggle/working/repo')

from config import Config
import torch

# Kaggle-spesifik path güncellemeleri
Config.TRAIN_PROCESSED_CSV = TRAIN_CSV
Config.VAL_PROCESSED_CSV   = VAL_CSV
Config.TEST_PROCESSED_CSV  = TEST_CSV
Config.IMAGE_DIR           = IMAGE_DIR

# Yeni kayıtların yapılacağı klasörler
Config.TRANSFORMER_CHECKPOINT_DIR = '/kaggle/working/checkpoints_transformer'
Config.LOG_DIR             = '/kaggle/working/logs'

# 🎯 EN ÖNEMLİ KISIM: RESUME (Kaldığı yerden devam)
Config.TRANSFORMER_RESUME = True
Config.TRANSFORMER_CHECKPOINT_FILE = LOCAL_CHECKPOINT_DIR

# 🎯 OOM MEMORY ERROR ÇÖZÜMÜ İÇİN:
# Encoder fine-tune aşamasında VRAM yetmez. 
# Batch'i 1'e veya en güvenli seviyeye çekip Gradient Accumulation ile toparlıyoruz.
Config.TRANSFORMER_BATCH_SIZE       = 1     # 4 veya 2 yerine 1! Memory yetmesini garantiler
Config.TRANSFORMER_GRAD_ACCUM_STEPS = 16    # 1 * 16 = 16 efektif batch size!

Config.TRANSFORMER_USE_AMP          = True  # FP16 açık
Config.NUM_WORKERS                  = 4
torch.backends.cudnn.benchmark      = True
Config.EPOCHS                       = 20

print('✅ Config RESUME ve OOM Önleme modu ile güncellendi!')
print(f'   Önceki checkpoint yolu: {Config.TRANSFORMER_CHECKPOINT_FILE}')
print(f'   Resim Başına Batch:     {Config.TRANSFORMER_BATCH_SIZE}')
print(f'   Grad accumulation:      {Config.TRANSFORMER_GRAD_ACCUM_STEPS}')
print(f'   Efektif batch:          {Config.TRANSFORMER_BATCH_SIZE * Config.TRANSFORMER_GRAD_ACCUM_STEPS}')

In [ ]:
# Eğitimi Başlat (Model yüklenecek ve Encoder unfreeze olup eğitime başlayacak)
import train_transformer
train_transformer.main()

In [ ]:
# Yeni checkpoint'leri kaydet
import shutil
shutil.make_archive(
    '/kaggle/working/new_best_model_transformer',
    'zip',
    '/kaggle/working/checkpoints_transformer/best_model'
)
print('✅ Yeni eğitimli model ZIP olarak kaydedildi! İndirebilirsin.')